# Lab 02: Model Selection & Customization

## Overview
In this lab you will fine-tune a foundation model on Amazon Bedrock, deploy a model via SageMaker JumpStart, and compare the fine-tuned model against its base version. You will learn when to fine-tune versus prompt engineer versus use RAG, and build a practical decision framework for model customization.

## Learning Objectives
- Prepare JSONL training data for Bedrock custom model training
- Create and monitor a Bedrock Custom Model Training (fine-tuning) job
- Configure LoRA-based fine-tuning on SageMaker JumpStart as an alternative path
- Evaluate fine-tuned model output against the base model on test questions
- Apply a model selection decision framework: prompt engineering → RAG → fine-tuning → continued pre-training

## Exam Domain
**Selection & Implementation of Foundation Models (26%)** + **Optimizing Performance (24%)** — This lab covers Task 1.3 (model training and customization) and Task 3.3 (fine-tuning for performance).

## Architecture Diagram
![Lab 02 Architecture](../assets/diagrams/lab-02-model-selection-customization.png)

In [ ]:
%pip install boto3 sagemaker --quiet

In [ ]:
import boto3, json, os, time

REGION = "us-east-1"

# Environment detection
if os.environ.get("AWS_DEFAULT_REGION") and os.path.exists("/opt/ml"):
    session = boto3.Session(region_name=REGION)
    print("Running in SageMaker Studio")
else:
    session = boto3.Session(region_name=REGION)
    print("Running locally")

bedrock = session.client("bedrock")
bedrock_runtime = session.client("bedrock-runtime")
s3 = session.client("s3")
sts = session.client("sts")

ACCOUNT_ID = sts.get_caller_identity()["Account"]
BUCKET = f"aws-genai-lab-{ACCOUNT_ID}"
BEDROCK_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/genai-lab-bedrock-role"

print(f"Account: {ACCOUNT_ID}")
print(f"Bucket:  {BUCKET}")
print(f"Role:    {BEDROCK_ROLE_ARN}")

## A. When to Customize (Decision Framework)

Before investing in model customization, you should follow this decision tree — starting with the cheapest and fastest approach and escalating only when needed:

1. **Prompt Engineering** (try first) — Craft better prompts with system instructions, few-shot examples, and chain-of-thought reasoning. This is the cheapest and fastest approach. If the model already knows the information but is not formatting or phrasing it the way you need, prompt engineering is usually sufficient.

2. **Retrieval-Augmented Generation (RAG)** — If the model lacks specific knowledge (e.g., your company's internal docs, recent data), add a retrieval layer. RAG injects relevant external context into the prompt at query time without modifying the model.

3. **Fine-Tuning** — If the model needs to change its behavior, style, tone, or output format in ways that prompt engineering cannot achieve, fine-tune it with labeled prompt-completion pairs. Fine-tuning modifies model weights to specialize it for your task.

4. **Continued Pre-Training** — If the model needs to learn an entirely new domain vocabulary or knowledge base (e.g., legal, medical, financial jargon), continued pre-training extends the model's foundational knowledge using unlabeled text.

### Comparison Table

| Approach | When to Use | Cost | Latency Impact | Examples |
|----------|-------------|------|----------------|----------|
| Prompt Engineering | Model has the knowledge, needs better output formatting | Lowest (no training) | None | System prompts, few-shot examples, chain-of-thought |
| RAG | Model lacks specific or recent knowledge | Low (vector DB + retrieval) | Slight increase (retrieval step) | Knowledge bases, document Q&A, support chatbots |
| Fine-Tuning | Model needs new behavior, style, or format | Medium ($5-50+ per job) | None (baked into weights) | Domain-specific tone, structured output, classification |
| Continued Pre-Training | Model needs entirely new domain knowledge | High ($100s+) | None (baked into weights) | Legal language, medical terminology, proprietary jargon |

## B. Prepare Training Data

Amazon Bedrock expects fine-tuning data in **JSONL format** — one JSON object per line, each with a `prompt` and `completion` field. The data must be stored in an S3 bucket that the Bedrock service role can access.

Key requirements for training data:
- **Format:** JSONL with `prompt` and `completion` keys (for Llama and Titan fine-tuning)
- **Minimum:** At least 100 examples recommended for meaningful fine-tuning
- **Quality:** High-quality, representative examples produce better results than large volumes of noisy data
- **Storage:** Must be uploaded to S3 before creating the training job

Let's load our training dataset, inspect its format, and upload it to S3.

In [ ]:
# Load and inspect training data
with open("data/finetune-qa.jsonl") as f:
    data = [json.loads(line) for line in f]

print(f"Total training examples: {len(data)}")
print(f"\nSample entry:\n{json.dumps(data[0], indent=2)}")

# Show a few more samples
print(f"\n--- Additional samples ---")
for i in [1, 2, 3]:
    print(f"\nExample {i+1}:")
    print(f"  Prompt:     {data[i]['prompt'][:80]}...")
    print(f"  Completion: {data[i]['completion'][:80]}...")

In [ ]:
# Upload training data to S3
s3.upload_file("data/finetune-qa.jsonl", BUCKET, "fine-tuning/train.jsonl")
print(f"Uploaded to s3://{BUCKET}/fine-tuning/train.jsonl")

# Verify the upload
response = s3.head_object(Bucket=BUCKET, Key="fine-tuning/train.jsonl")
print(f"File size: {response['ContentLength']} bytes")

## C. Fine-Tune on Amazon Bedrock

Amazon Bedrock Custom Model Training lets you fine-tune a base foundation model using your own labeled data. The process:

1. **Upload training data** to S3 (done in Section B)
2. **Create a customization job** specifying the base model, training data location, output location, and hyperparameters
3. **Wait for training** to complete (typically 30-60+ minutes depending on dataset size and model)
4. **Purchase provisioned throughput** for the custom model to make it available for inference

Key hyperparameters for fine-tuning:
- **epochCount** — Number of complete passes through the training data. More epochs can improve learning but risk overfitting.
- **batchSize** — Number of training examples processed per update step. Smaller batches use less memory but train more slowly.
- **learningRate** — Controls how much the model weights change per update. Too high causes instability, too low causes slow learning.

Let's create the fine-tuning job.

In [ ]:
import datetime

job_name = f"genai-lab-finetune-{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}"

job = bedrock.create_model_customization_job(
    jobName=job_name,
    customModelName="genai-lab-custom-llama",
    roleArn=BEDROCK_ROLE_ARN,
    baseModelIdentifier="meta.llama3-8b-instruct-v1:0",
    customizationType="FINE_TUNING",
    trainingDataConfig={"s3Uri": f"s3://{BUCKET}/fine-tuning/train.jsonl"},
    outputDataConfig={"s3Uri": f"s3://{BUCKET}/fine-tuning/output/"},
    hyperParameters={
        "epochCount": "1",
        "batchSize": "1",
        "learningRate": "0.00001"
    }
)
print(f"Training job created: {job['jobArn']}")
print(f"Job name: {job_name}")

### Poll Training Status

Fine-tuning typically takes **30-60+ minutes** depending on the dataset size and base model. The cell below polls for status every 60 seconds. You can:
- Let it run and continue reading the next sections while you wait
- Skip ahead to **Section D** (SageMaker JumpStart) which is an independent alternative path
- Come back later and check status manually

In [ ]:
# Poll for training job completion
while True:
    status = bedrock.get_model_customization_job(jobIdentifier=job["jobArn"])
    current_status = status["status"]
    print(f"Status: {current_status} ({datetime.datetime.now().strftime('%H:%M:%S')})")
    if current_status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(60)

if current_status == "Completed":
    print(f"\nCustom model ARN: {status['outputModelArn']}")
    print("Next step: purchase provisioned throughput to enable inference")
elif current_status == "Failed":
    print(f"\nFailure reason: {status.get('failureMessage', 'Unknown')}")

## D. Deploy via SageMaker JumpStart (Alternative Path)

**SageMaker JumpStart** is a machine learning hub that provides pre-trained foundation models you can deploy to SageMaker endpoints. It offers an alternative to Bedrock for model deployment with more infrastructure control.

### Bedrock vs SageMaker JumpStart

| Feature | Amazon Bedrock | SageMaker JumpStart |
|---------|---------------|---------------------|
| Infrastructure | Fully managed, no instances to configure | You choose instance types and manage endpoints |
| Pricing | Pay per token (on-demand) or provisioned throughput | Pay per hour for the endpoint instance |
| Customization | Fine-tuning via managed jobs | Full control over training scripts, LoRA, QLoRA |
| Scaling | Automatic (on-demand) or reserved capacity | Manual or auto-scaling policies on endpoints |
| Best for | Production APIs, managed simplicity | Research, custom training pipelines, full control |

Below we configure a JumpStart model. The `deploy()` call is commented out to avoid creating a billable endpoint (~$1.50/hr for ml.g5.2xlarge). Uncomment it if you want to deploy and test.

In [ ]:
from sagemaker.jumpstart.model import JumpStartModel

model = JumpStartModel(
    model_id="meta-textgeneration-llama-3-8b-instruct",
    role=BEDROCK_ROLE_ARN,
)

# Note: This creates a real-time endpoint (~$1.50/hr for ml.g5.2xlarge)
# Uncomment to deploy:
# predictor = model.deploy(initial_instance_count=1, instance_type="ml.g5.2xlarge")
print("JumpStart model configured (deploy uncommented to save cost)")
print(f"Model ID: {model.model_id}")
print(f"Instance type: ml.g5.2xlarge (~$1.50/hr)")
print(f"\nTo deploy, uncomment the predictor.deploy() line above and re-run this cell.")

## E. Compare Base vs Fine-Tuned Model

After the fine-tuning job completes (Section C), you can compare the custom model's responses against the base model. This evaluation step is critical — fine-tuning does not always improve results, and you should verify that the custom model actually performs better on your target task.

**Important:** Custom models in Bedrock require **provisioned throughput** for inference. On-demand pricing is not available for custom models. If you have not purchased provisioned throughput, the fine-tuned model calls below will return an error — the base model comparison will still work.

The function below sends the same test questions to both models and displays side-by-side results.

In [ ]:
test_questions = [
    "What is Amazon Bedrock Knowledge Bases?",
    "When should you use provisioned throughput?",
    "What are Bedrock Guardrails?",
]

def compare_models(base_id, custom_id, questions):
    for q in questions:
        print(f"\nQuestion: {q}")
        print("-" * 60)

        # Base model
        base_resp = bedrock_runtime.converse(
            modelId=base_id,
            messages=[{"role": "user", "content": [{"text": q}]}],
            inferenceConfig={"maxTokens": 200}
        )
        print(f"Base: {base_resp['output']['message']['content'][0]['text'][:200]}")

        # Custom model (requires provisioned throughput)
        try:
            custom_resp = bedrock_runtime.converse(
                modelId=custom_id,
                messages=[{"role": "user", "content": [{"text": q}]}],
                inferenceConfig={"maxTokens": 200}
            )
            print(f"Fine-tuned: {custom_resp['output']['message']['content'][0]['text'][:200]}")
        except Exception as e:
            print(f"Fine-tuned: (not ready yet — {e})")

        print()

# Run comparison
compare_models(
    base_id="meta.llama3-8b-instruct-v1:0",
    custom_id="genai-lab-custom-llama",
    questions=test_questions
)

## Cleanup

Run the cell below to delete resources created in this lab. Custom models and SageMaker endpoints incur ongoing charges if left running.

In [ ]:
# Delete custom model (if created)
try:
    bedrock.delete_custom_model(modelIdentifier="genai-lab-custom-llama")
    print("Custom model deleted")
except Exception as e:
    print(f"Skip: {e}")

# Delete SageMaker endpoint (if deployed)
# predictor.delete_endpoint()
# print("SageMaker endpoint deleted")

# Delete training data from S3
try:
    s3.delete_object(Bucket=BUCKET, Key="fine-tuning/train.jsonl")
    print("Training data deleted from S3")
except Exception as e:
    print(f"Skip: {e}")

print("\nCleanup complete.")

## Key Takeaways

1. Always start with **prompt engineering** before considering fine-tuning — it is the cheapest and fastest approach and often sufficient
2. Use **RAG** when the model needs access to external or recent knowledge; use **fine-tuning** when the model needs to change its behavior, style, or output format
3. **Training data quality** matters more than quantity — a small, high-quality dataset produces better results than a large, noisy one
4. Always **evaluate fine-tuned models** against the base model on representative test cases before deploying to production
5. Fine-tuned models on Bedrock require **provisioned throughput** for inference, which adds ongoing cost — factor this into your total cost of ownership

## Key Concepts

| Concept | Definition |
|---------|-----------|
| Fine-Tuning | Customizing a foundation model using labeled prompt-completion pairs to improve performance on a specific task, style, or format |
| Continued Pre-Training | Extending a model's domain knowledge by training on additional unlabeled text data, without targeting a specific task format |
| LoRA (Low-Rank Adaptation) | A parameter-efficient fine-tuning technique that adds small trainable matrices to the model instead of updating all weights, reducing compute and memory requirements |
| PEFT (Parameter-Efficient Fine-Tuning) | A family of techniques (including LoRA, QLoRA, adapters) that fine-tune only a small subset of model parameters, making customization faster and cheaper |
| Custom Model Training | The Bedrock feature that runs a managed fine-tuning or continued pre-training job, producing a private custom model stored in your account |
| SageMaker JumpStart | A SageMaker hub providing pre-trained foundation models that can be deployed to managed endpoints with full infrastructure control |
| Hyperparameters | Settings that control the training process (learning rate, epoch count, batch size) rather than being learned from the data |

## Exam Preparation — Common Exam Question Patterns

**Q: When should you fine-tune a model instead of using prompt engineering?**
A: Fine-tune when you need the model to consistently adopt a new behavior, style, tone, or output format that cannot be achieved through prompt engineering alone. Examples include making a model always respond in a specific domain terminology or consistently producing structured JSON output.

**Q: What is the difference between fine-tuning and continued pre-training?**
A: Fine-tuning uses labeled prompt-completion pairs to teach a model a specific task or format. Continued pre-training uses unlabeled domain text to expand the model's foundational knowledge. Use fine-tuning for task specialization, continued pre-training for domain knowledge.

**Q: How do you deploy a custom fine-tuned model on Amazon Bedrock for inference?**
A: After fine-tuning completes, you must purchase provisioned throughput for the custom model. Custom models cannot use on-demand pricing — only provisioned throughput with a 1-month or 6-month commitment term makes them available for inference.

**Q: What is LoRA and why is it useful for fine-tuning?**
A: LoRA (Low-Rank Adaptation) is a parameter-efficient fine-tuning technique that adds small trainable matrices to the model instead of updating all weights. It reduces compute costs, memory requirements, and training time while achieving comparable results to full fine-tuning.

**Q: When should you use SageMaker JumpStart instead of Amazon Bedrock?**
A: Use SageMaker JumpStart when you need full control over the training and deployment infrastructure, want to use custom training scripts or advanced techniques (LoRA, QLoRA), or need to deploy models on specific instance types. Use Bedrock when you want a fully managed experience with no infrastructure management.

## Cost Breakdown

| Resource | Usage | Est. Cost |
|----------|-------|-----------|
| Bedrock Fine-Tuning Job (Llama 3 8B) | 1 epoch, ~110 examples | ~$5-10 |
| SageMaker Endpoint (if deployed) | ml.g5.2xlarge, per hour | ~$1.50/hr |
| S3 Storage | Training data + model artifacts | Negligible |
| Bedrock API (base model comparison) | ~3K input + 3K output tokens | ~$0.10 |
| **Total** | | **~$8-12** |

**Cost-saving tips:**
- The SageMaker endpoint is commented out by default — only deploy it if you want to test JumpStart
- Delete the custom model promptly after evaluation to stop provisioned throughput charges
- The fine-tuning job itself is a one-time cost; ongoing costs come from provisioned throughput for inference